In [1]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

import regrid as rg
from load_cmip_esgf import CMIPESGFLoader

# Check manifest

In [ ]:
ROOTDIR = Path("/glade/derecho/scratch/bbuchovecky/cmip_intake_esgf_fetch/manifests")

In [11]:
cat = pd.read_csv(ROOTDIR / "cmip_intake_manifest_selected.csv")

In [12]:
cat["source_id"].unique()

<ArrowStringArray>
[  'AWI-ESM-1-REcoM',       'ICON-ESM-LR',         'CMCC-ESM2',
        'CAS-ESM2-0',       'FIO-ESM-2-0',      'CMCC-CM2-HR4',
           'NorCPM1',           'TaiESM1',      'CMCC-CM2-SR5',
     'AWI-CM-1-1-MR',    'AWI-ESM-1-1-LR',          'IITM-ESM',
       'GISS-E2-2-G',       'GISS-E2-2-H',         'CESM2-FV2',
   'CESM2-WACCM-FV2',     'ACCESS-ESM1-5',        'ACCESS-CM2',
        'NorESM2-MM',         'FGOALS-g3',    'GISS-E2-1-G-CC',
        'NorESM2-LM',        'MCM-UA-1-0',     'MPI-ESM1-2-HR',
     'MPI-ESM1-2-LR',       'CAMS-CSM1-0',             'NESM3',
   'MPI-ESM-1-2-HAM',         'CanESM5-1',           'CanESM5',
       'GISS-E2-1-H',       'SAM0-UNICON',            'MIROC6',
             'CESM2',       'CESM2-WACCM',        'MRI-ESM2-0',
          'BCC-ESM1',       'BCC-CSM2-MR',       'GISS-E2-1-G',
       'UKESM1-1-LL',        'MIROC-ES2H',        'MIROC-ES2L',
       'UKESM1-0-LL',   'HadGEM3-GC31-MM',   'HadGEM3-GC31-LL',
     'CanESM5-CanOE',

In [24]:
for sid in sorted(cat["source_id"].unique()):
    print(sid, end=": ")
    for vid in cat["variable_id"].unique():
        if vid not in list(cat.loc[cat["source_id"] == sid]["variable_id"]) and vid != "areacella":
            print(vid, end=", ")
        # else:
        #     print("✅", end=", ")
    print()

ACCESS-CM2: lai, 
ACCESS-ESM1-5: 
AWI-CM-1-1-MR: lai, 
AWI-ESM-1-1-LR: 
AWI-ESM-1-REcoM: 
BCC-CSM2-MR: 
BCC-ESM1: 
CAMS-CSM1-0: 
CAS-ESM2-0: 
CESM2: 
CESM2-FV2: 
CESM2-WACCM: 
CESM2-WACCM-FV2: 
CIESM: evspsbl, 
CMCC-CM2-HR4: 
CMCC-CM2-SR5: 
CMCC-ESM2: 
CNRM-CM6-1: 
CNRM-CM6-1-HR: 
CNRM-ESM2-1: 
CanESM5: 
CanESM5-1: 
CanESM5-CanOE: 
E3SM-1-0: 
E3SM-1-1: 
E3SM-1-1-ECA: 
E3SM-2-0: 
E3SM-2-0-NARRM: 
E3SM-2-1: 
EC-Earth3: lai, 
EC-Earth3-AerChem: lai, 
EC-Earth3-CC: 
EC-Earth3-ESM-1: 
EC-Earth3-HR: lai, rlds, rlus, 
EC-Earth3-Veg: 
EC-Earth3-Veg-LR: 
FGOALS-f3-L: lai, 
FGOALS-g3: 
FIO-ESM-2-0: 
GFDL-CM4: evspsbl, pr, rlds, rlus, rsds, rsus, 
GFDL-ESM4: 
GISS-E2-1-G: 
GISS-E2-1-G-CC: 
GISS-E2-1-H: 
GISS-E2-2-G: 
GISS-E2-2-H: 
GISS-E3-G: 
HadGEM3-GC31-LL: 
HadGEM3-GC31-MM: 
ICON-ESM-LR: lai, 
IITM-ESM: lai, 
INM-CM4-8: 
INM-CM5-0: 
IPSL-CM5A2-INCA: 
IPSL-CM6A-LR: 
IPSL-CM6A-LR-INCA: 
KACE-1-0-G: lai, 
KIOST-ESM: evspsbl, 
MCM-UA-1-0: lai, rlds, rlus, rsds, rsus, 
MIROC-ES2H: 
MIROC-ES2L: 
MIR

# Load catalog

In [3]:
CATALOG_PATH = Path("/glade/derecho/scratch/bbuchovecky/cmip_intake_esgf_fetch/catalogs/cmip_evap.csv")

VARIABLES = ["evspsbl", "lai", "pr", "rlds", "rlus", "rsds", "rsus"]
EXPERIMENT_ID = "historical"
SOURCE_IDS = None
MEMBER_IDS = None
TIME_SLICE = slice("1950-01", "2014-12")

In [4]:
loader = CMIPESGFLoader(CATALOG_PATH)
catalog = loader.catalog

In [7]:
sid_all_avail = list(sorted(catalog["source_id"].unique()))

for sid in sorted(catalog["source_id"].unique()):
    for vid in catalog["variable_id"].unique():
        if vid not in list(catalog.loc[catalog["source_id"] == sid]["variable_id"]) and vid != "areacella":
            print(sid)
            sid_all_avail.remove(sid)
            break

print(f"Total models: {len(catalog['source_id'].unique())}")
print(f"Models with all variables: {len(sid_all_avail)}")

ACCESS-CM2
AWI-CM-1-1-MR
CIESM
EC-Earth3
EC-Earth3-AerChem
EC-Earth3-HR
FGOALS-f3-L
GFDL-CM4
ICON-ESM-LR
IITM-ESM
KACE-1-0-G
KIOST-ESM
MCM-UA-1-0
MIROC6
NESM3
NorCPM1
Total models: 74
Models with all variables: 58


In [28]:
print("Load data:")
load_t0 = time.perf_counter()
data_dict = loader.load_data(
    VARIABLES,
    source_id=sid_all_avail,
    member_id="top",
    experiment_id=EXPERIMENT_ID,
    time_slice=TIME_SLICE
)
load_elapsed = time.perf_counter() - load_t0
print(f"  loaded in {load_elapsed:.2f}s")

Load data:
only selecting top member_id
=== ACCESS-ESM1-5 ===
  evspsbl      n=1     ✅r1i1p1f1 
  lai          n=1     ✅r1i1p1f1 
  pr           n=1     ✅r1i1p1f1 
  rlds         n=1     ✅r1i1p1f1 
  rlus         n=1     ✅r1i1p1f1 
  rsds         n=1     ✅r1i1p1f1 
  rsus         n=1     ✅r1i1p1f1 
=== AWI-ESM-1-1-LR ===
  evspsbl      n=1     ✅r1i1p1f1 
  lai          n=1     ✅r1i1p1f1 
  pr           n=1     ✅r1i1p1f1 
  rlds         n=1     ✅r1i1p1f1 
  rlus         n=1     ✅r1i1p1f1 
  rsds         n=1     ✅r1i1p1f1 
  rsus         n=1     ✅r1i1p1f1 
=== AWI-ESM-1-REcoM ===
  evspsbl      n=1     ✅r1i1p1f1 
  lai          n=1     ✅r1i1p1f1 
  pr           n=1     ✅r1i1p1f1 
  rlds         n=1     ✅r1i1p1f1 
  rlus         n=1     ✅r1i1p1f1 
  rsds         n=1     ✅r1i1p1f1 
  rsus         n=1     ✅r1i1p1f1 
=== BCC-CSM2-MR ===
  evspsbl      n=1     ✅r1i1p1f1 
  lai          n=1     ✅r1i1p1f1 
  pr           n=1     

KeyboardInterrupt: 